# DF-1 — Deepfake Video Detection

Detect face swaps, reenactments, and other facial manipulations in video content.

**Model:** `DF-1`  
**Input:** Video files (`.mp4`, `.mov`, etc.)

---
### Contents
1. Setup
2. One-call detection
3. Inspect per-participant results
4. Two-step: upload now, poll later
5. Visualize results — heatmap & bounding boxes
6. Error handling
7. Async client

---
## 1. Setup

In [ ]:
import sys
import os

sys.path.insert(0, os.path.abspath('../src'))

from authenta.authenta_client import AuthentaClient
from authenta import (
    AuthentaError,
    AuthenticationError,
    QuotaExceededError,
    InsufficientCreditsError,
)

CLIENT_ID     = "YOUR_CLIENT_ID"
CLIENT_SECRET = "YOUR_CLIENT_SECRET"
BASE_URL      = "https://platform.authenta.ai"

client = AuthentaClient(
    base_url=BASE_URL,
    client_id=CLIENT_ID,
    client_secret=CLIENT_SECRET,
)

print("Client ready.")

---
## 2. One-call Detection

`client.process()` uploads the file and blocks until the result is ready.

In [ ]:
VIDEO = "../data_samples/fake1.mp4"

media = client.process(VIDEO, model_type="DF-1")

print(f"Media ID     : {media['mid']}")
print(f"Status       : {media['status']}")
print(f"Is Fake      : {media.get('fake')}")
print(f"Result URL   : {media.get('resultURL')}")
print(f"Participants : {len(media.get('participants', []))} face(s) detected")

---
## 3. Inspect Per-Participant Results

DF-1 tracks each detected face (participant) separately, with individual confidence scores and heatmap URLs.

In [ ]:
for idx, p in enumerate(media.get("participants", [])):
    print(f"\nParticipant {idx}")
    print(f"  Fake        : {p.get('fake')}")
    print(f"  Confidence  : {p.get('confidence')}")
    print(f"  Heatmap URL : {p.get('heatmapURL')}")

---
## 4. Two-step: Upload Now, Poll Later

Upload the video immediately and collect the result later — useful for long videos where you don't want to block.

In [ ]:
# Step 1 — upload
upload_meta = client.upload_file(VIDEO, model_type="DF-1")
mid = upload_meta["mid"]
print(f"Uploaded.  Media ID : {mid}")
print(f"           Status   : {upload_meta['status']}")

In [ ]:
# Step 2 — poll (15-minute timeout for longer videos)
media = client.wait_for_media(mid, interval=10.0, timeout=900.0)
print(f"Status  : {media['status']}")
print(f"Is Fake : {media.get('fake')}")

---
## 5. Visualize Results

### 5a. Per-participant heatmap videos

In [ ]:
from authenta.visualization import save_heatmap

media = client.process(VIDEO, model_type="DF-1")

os.makedirs("results", exist_ok=True)

# Saves one heatmap video per participant:
#   results/heatmap_p0.mp4, results/heatmap_p1.mp4, ...
save_heatmap(
    media=media,
    out_path="./results",
    model_type="DF-1",
)
print("Heatmap video(s) saved to ./results/")

### 5b. Bounding box annotated video

In [ ]:
from authenta.visualization import save_bounding_box_video

save_bounding_box_video(
    media,
    src_video_path=VIDEO,
    out_video_path="results/df1_annotated.mp4",
)
print("Annotated video saved to results/df1_annotated.mp4")

---
## 6. Error Handling

In [ ]:
try:
    media = client.process(VIDEO, model_type="DF-1")
    print(f"Is Fake: {media.get('fake')}")
except AuthenticationError:
    print("Authentication failed — check your CLIENT_ID and CLIENT_SECRET.")
except QuotaExceededError:
    print("API quota exceeded — upgrade your plan.")
except InsufficientCreditsError:
    print("Not enough credits.")
except TimeoutError as e:
    print(f"Timed out: {e}")
except AuthentaError as e:
    print(f"API error [{e.code}]: {e.message}")

---
## 7. Async Client

In [ ]:
import asyncio
from authenta.async_authenta_client import AsyncAuthentaClient

# One-call — async
async def detect_single():
    async with AsyncAuthentaClient(
        base_url=BASE_URL,
        client_id=CLIENT_ID,
        client_secret=CLIENT_SECRET,
    ) as client:
        media = await client.process(VIDEO, model_type="DF-1")
        print(f"Status  : {media['status']}")
        print(f"Is Fake : {media.get('fake')}")

await detect_single()

In [ ]:
# Two-step — async
async def detect_two_step():
    async with AsyncAuthentaClient(
        base_url=BASE_URL,
        client_id=CLIENT_ID,
        client_secret=CLIENT_SECRET,
    ) as client:
        upload_meta = await client.upload_file(VIDEO, model_type="DF-1")
        mid = upload_meta["mid"]
        print(f"Uploaded: {mid}")

        media = await client.wait_for_media(mid, interval=10.0, timeout=900.0)
        print(f"Status  : {media['status']}")
        print(f"Is Fake : {media.get('fake')}")

await detect_two_step()